<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [1]:
# Google Colab usually includes pandas and folium.
# Run this cell if an installation/update is needed (safe in Colab).
%pip -q install folium pandas requests

In [2]:
import folium
import pandas as pd
import requests
from html import escape

In [3]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


In [4]:
## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [5]:
# Google Colab compatible: read the CSV URL directly with pandas.
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
spacex_df = pd.read_csv(URL)
print('Dataset loaded:', spacex_df.shape)
display(spacex_df.head())

Dataset loaded: (56, 13)


,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


Now, you can take a look at what are the coordinates for each site.


In [6]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [7]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [8]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [9]:
# TASK 1 — circles and text labels for every unique launch site.
site_map = folium.Map(location=[38.0, -99.0], zoom_start=4)

for _, site in launch_sites_df.iterrows():
    location = [float(site['Lat']), float(site['Long'])]
    site_name = str(site['Launch Site'])
    folium.Circle(
        location=location, radius=900, color='#D97706',
        fill=True, fill_color='#F59E0B', fill_opacity=0.3,
        popup=folium.Popup(escape(site_name), max_width=300),
        tooltip=site_name,
    ).add_to(site_map)
    folium.Marker(
        location=location,
        icon=DivIcon(
            icon_size=(180, 35), icon_anchor=(0, 0),
            html=f'<div style="font-size:11px;color:#9A3412;font-weight:bold;white-space:nowrap;">{escape(site_name)}</div>',
        ),
    ).add_to(site_map)

site_map.save('spacex_launch_sites_map.html')
print('Saved spacex_launch_sites_map.html')
site_map

Saved spacex_launch_sites_map.html


The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


**Geographic note:** The listed US launch sites are in Florida and California: they are on or near the coast, but they are not on the equator. Proximity alone cannot establish a causal effect on landing success.

In [10]:
# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [11]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [12]:
marker_cluster = MarkerCluster()


*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


In [13]:
# TASK 2 — 1 = success (green), 0 = failed/non-success (red).
# Keep the field name expected by the IBM lab.
spacex_df['marker_color'] = spacex_df['class'].apply(
    lambda result: 'green' if int(result) == 1 else 'red'
)
print(spacex_df[['Launch Site', 'class', 'marker_color']].head(10).to_string(index=False))

Launch Site  class marker_color
CCAFS LC-40      0          red
CCAFS LC-40      0          red
CCAFS LC-40      0          red
CCAFS LC-40      0          red
CCAFS LC-40      0          red
CCAFS LC-40      0          red
CCAFS LC-40      0          red
CCAFS LC-40      0          red
CCAFS LC-40      0          red
CCAFS LC-40      0          red


*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [14]:
# TASK 2 — Add each launch outcome to the marker cluster.
site_map.add_child(marker_cluster)
for _, record in spacex_df.iterrows():
    success = int(record['class']) == 1
    label = 'Successful landing' if success else 'Unsuccessful landing'
    marker = folium.Marker(
        location=[float(record['Lat']), float(record['Long'])],
        popup=f"{escape(str(record['Launch Site']))}: {label}",
        tooltip=label,
        icon=folium.Icon(color=record['marker_color'], icon='info-sign'),
    )
    marker.add_to(marker_cluster)

site_map.save('spacex_launch_outcomes_map.html')
print('Saved spacex_launch_outcomes_map.html')
site_map

Saved spacex_launch_outcomes_map.html


Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


In [15]:
# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [16]:
# TASK 3 — choose a site for the proximity study.
# The coastline example in the IBM lab is near CCAFS, so use that site.
ccafs = launch_sites_df[
    launch_sites_df['Launch Site'].astype(str).str.contains('CCAFS', case=False, na=False)
]
selected_site = ccafs.iloc[0] if len(ccafs) else launch_sites_df.iloc[0]
launch_site_name = str(selected_site['Launch Site'])
launch_site_lat = float(selected_site['Lat'])
launch_site_lon = float(selected_site['Long'])
launch_site_coord = [launch_site_lat, launch_site_lon]

proximity_map = folium.Map(location=launch_site_coord, zoom_start=11)
folium.Marker(
    launch_site_coord, tooltip=launch_site_name,
    popup=escape(launch_site_name), icon=folium.Icon(color='blue', icon='info-sign')
).add_to(proximity_map)

formatter = 'function(num) {return L.Util.formatNum(num, 5);};'
MousePosition(
    position='topright', separator=' Long: ', empty_string='NaN',
    lng_first=False, num_digits=20, prefix='Lat:',
    lat_formatter=formatter, lng_formatter=formatter,
).add_to(proximity_map)
# Click on the map to see the exact coordinates to reuse below.
from folium.features import LatLngPopup
LatLngPopup().add_to(proximity_map)
print('Selected launch site:', launch_site_name, launch_site_coord)
proximity_map

Selected launch site: CCAFS LC-40 [28.56230197, -80.57735648]


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [17]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [18]:
# TASK 3 — example coastline point supplied by the IBM lab.
# It is an illustrative point, not a mathematically verified closest point.
coastline_lat, coastline_lon = 28.56367, -80.57163
if 'CCAFS' not in launch_site_name.upper():
    print('Warning: IBM coastline sample is near CCAFS; for this site, select a coastal point using MousePosition.')
coastline_coord = [coastline_lat, coastline_lon]
distance_coastline = calculate_distance(
    launch_site_lat, launch_site_lon, coastline_lat, coastline_lon
)
print(f'Distance to selected coastline sample: {distance_coastline:.2f} km')

Distance to selected coastline sample: 0.58 km


In [19]:
# Add the coastline marker and show its straight-line distance.
coastline_label = f'Coastline sample: {distance_coastline:.2f} km (straight line)'
folium.Marker(
    coastline_coord,
    tooltip=coastline_label,
    popup=coastline_label,
    icon=folium.Icon(color='orange', icon='info-sign'),
).add_to(proximity_map)
folium.Marker(
    coastline_coord,
    icon=DivIcon(
        icon_size=(175, 25), icon_anchor=(0, 0),
        html=(f'<div style="font-size:12px;color:#C2410C;font-weight:bold;white-space:nowrap;">'
              f'{distance_coastline:.2f} km</div>'),
    ),
).add_to(proximity_map)
proximity_map

*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point


In [20]:
# Connect the launch site and sample coastline point with a PolyLine.
lines = folium.PolyLine(
    locations=[launch_site_coord, coastline_coord],
    weight=3, color='#EA580C', opacity=0.85,
    tooltip=f'Coastline sample: {distance_coastline:.2f} km',
)
proximity_map.add_child(lines)
proximity_map

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


**Note:** The next cell uses OpenStreetMap's Overpass API for optional nearby railway, major road and city points. These are approximations based on mapped feature centers, not verified nearest geometric points. If the API is unavailable, use MousePosition and fill the `manual_points` dictionary with points you have located on the map.

A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [21]:
# TASK 3 — optional, data-grounded proximity to nearby cities, railway, highway.
# Query OpenStreetMap's Overpass API; never fabricate feature coordinates.
# OSM way centers are approximate representatives, not the nearest point on a way.
# If Overpass is unavailable, the map still works and you can use MousePosition.

def discover_osm_proximities(lat, lon, radius_m=18000):
    query = f'''[out:json][timeout:18];
    (
      node(around:{radius_m},{lat},{lon})[place~"^(city|town)$"];
      way(around:{radius_m},{lat},{lon})[railway="rail"];
      way(around:{radius_m},{lat},{lon})[highway~"^(motorway|trunk|primary)$"];
    );
    out center 150;'''
    response = requests.post(
        'https://overpass-api.de/api/interpreter',
        data={'data': query}, timeout=23,
    )
    response.raise_for_status()
    choices = {'City / town': [], 'Railway': [], 'Major road': []}
    for feature in response.json().get('elements', []):
        tags = feature.get('tags', {})
        center = feature.get('center', feature)
        if 'lat' not in center or 'lon' not in center:
            continue
        if tags.get('place') in ('city', 'town'):
            category = 'City / town'
        elif tags.get('railway') == 'rail':
            category = 'Railway'
        elif tags.get('highway') in ('motorway', 'trunk', 'primary'):
            category = 'Major road'
        else:
            continue
        d = calculate_distance(lat, lon, center['lat'], center['lon'])
        choices[category].append((d, center['lat'], center['lon'], tags.get('name', category)))
    return {category: min(points) for category, points in choices.items() if points}

# One request for all three categories. Network-dependent; failures are nonfatal.
try:
    nearby_points = discover_osm_proximities(launch_site_lat, launch_site_lon)
    print('OSM categories located:', ', '.join(nearby_points) or 'none')
except (requests.RequestException, ValueError, KeyError) as err:
    nearby_points = {}
    print('OSM service unavailable; skip automatic proximity markers:', str(err)[:200])

# Manual override: once you inspect the map, you may supply measured coordinates.
# Do not fill with guessed coordinates.
manual_points = {}  # e.g. {'Major road': (28.5, -80.6)} — replace with observed map coordinates
for category, coords in manual_points.items():
    plat, plon = coords
    d = calculate_distance(launch_site_lat, launch_site_lon, plat, plon)
    nearby_points[category] = (d, plat, plon, 'Manually selected map point')

for category, (distance, plat, plon, name) in nearby_points.items():
    target = [plat, plon]
    popup_text = f'{category}: {name}; approx. straight-line distance: {distance:.2f} km'
    folium.Marker(target, popup=escape(popup_text), tooltip=popup_text,
                  icon=folium.Icon(color='purple', icon='info-sign')).add_to(proximity_map)
    folium.PolyLine([launch_site_coord, target], weight=2, color='#7C3AED',
                    dash_array='5,6', tooltip=f'{category}: ~{distance:.2f} km').add_to(proximity_map)

if nearby_points:
    display(pd.DataFrame([
        {'Feature': k, 'Distance (km)': round(v[0], 2), 'Name': v[3],
         'Latitude': v[1], 'Longitude': v[2]}
        for k, v in nearby_points.items()
    ]))
else:
    print('Use MousePosition to pick accurate city/railway/highway points, then rerun with manual_points.')
proximity_map

OSM service unavailable; skip automatic proximity markers: 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter
Use MousePosition to pick accurate city/railway/highway points, then rerun with manual_points.


In [22]:
# Save all proximity markers and their distance lines as an HTML file.
proximity_map.save('spacex_launch_site_proximities.html')
print('Saved spacex_launch_site_proximities.html')
print('Distances above are straight-line distances, not road/travel distances.')

Saved spacex_launch_site_proximities.html
Distances above are straight-line distances, not road/travel distances.


In [23]:
# View the final map. Take a screenshot for the IBM capstone presentation.
proximity_map

After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
